In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 3, Finished, Available, Finished, False)

In [2]:
# ============================================================
# CELL 1 — taxi_zone_lookup yuklab olish
# ============================================================
import requests, io, pandas as pd
from pyspark.sql import functions as F

# TLC rasmiy CSV
resp = requests.get(
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
)
zone_pdf = pd.read_csv(io.StringIO(resp.text))
zone_pdf.columns = ["location_id", "borough", "zone_name", "service_zone"]

zone_df = spark.createDataFrame(zone_pdf) \
    .withColumn("location_id", F.col("location_id").cast("int"))

print(f"✅ Zone lookup: {zone_df.count()} qator")
zone_df.show(5)

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 4, Finished, Available, Finished, False)

✅ Zone lookup: 265 qator
+-----------+-------------+--------------------+------------+
|location_id|      borough|           zone_name|service_zone|
+-----------+-------------+--------------------+------------+
|          1|          EWR|      Newark Airport|         EWR|
|          2|       Queens|         Jamaica Bay|   Boro Zone|
|          3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|          4|    Manhattan|       Alphabet City| Yellow Zone|
|          5|Staten Island|       Arden Heights|   Boro Zone|
+-----------+-------------+--------------------+------------+
only showing top 5 rows



In [3]:
# ============================================================
# CELL 2 — Taxi + Zone lookup → borough qo'shish
# ============================================================
taxi = spark.read.table("taxi_silver_clean")

taxi_with_borough = taxi \
    .withColumn(
        "pickup_date",
        F.to_date(F.col("tpep_pickup_datetime").cast("timestamp"))
    ) \
    .join(
        zone_df.select("location_id", "borough", "zone_name"),
        F.col("PULocationID") == F.col("location_id"),
        "left"
    ) 

taxi_with_borough.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("taxi_with_borough")
display(taxi_with_borough.limit(10))

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a91e1948-aa46-4fc7-ad41-9ad741459fa8)

In [4]:
taxi_with_borough.select("borough").distinct().show()

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 6, Finished, Available, Finished, False)

+-------------+
|      borough|
+-------------+
|       Queens|
|          EWR|
|      Unknown|
|     Brooklyn|
|Staten Island|
|    Manhattan|
|        Bronx|
|         NULL|
+-------------+



In [5]:
# ============================================================
# CELL — OpenAQ ga borough qo'shish (sensor_id bo'yicha)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. bronze_openaq_locations dan sensor + koordinata + borough
locations = spark.read.table("bronze_openaq_locations")

def get_borough(lat, lon):
    if lat is None or lon is None:
        return "Unknown"
    
    # Staten Island: 40.5795° N, -74.1502° W
    if 40.49 <= lat <= 40.65 and -74.25 <= lon <= -74.05:
        return "Staten Island"
    
    # Brooklyn: 40.6782° N, -73.9442° W
    if 40.57 <= lat <= 40.71 and -74.05 <= lon <= -73.83:
        return "Brooklyn"
    
    # Manhattan: 40.7831° N, -73.9712° W
    if 40.70 <= lat <= 40.88 and -74.02 <= lon <= -73.93:
        return "Manhattan"
    
    # Queens: 40.7282° N, -73.7949° W
    if 40.54 <= lat <= 40.80 and -73.93 <= lon <= -73.70:
        return "Queens"
    
    # Bronx: 40.8370° N, -73.8654° W
    if 40.80 <= lat <= 40.92 and -73.94 <= lon <= -73.74:
        return "Bronx"
    
    # EWR - Newark Liberty International Airport (NJ)
    if 40.67 <= lat <= 40.71 and -74.20 <= lon <= -74.15:
        return "EWR"
    
    return "Unknown"

borough_udf = F.udf(get_borough, StringType())

locations_with_borough = locations.withColumn(
    "borough", borough_udf("latitude", "longitude")
)

display(locations_with_borough.limit(100))

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fac0a86e-037a-4af4-9382-c1c9db06b5a3)

In [6]:
from pyspark.sql import functions as F

silver = spark.read.table("silver_openaq_daily_2025")

# locations dan faqat kerakli ustunlar
loc_slim = locations_with_borough.select(
    F.col("sensor_id").alias("loc_sensor_id"),
    "name",
    "borough",
    "latitude",
    "longitude"
).dropDuplicates(["loc_sensor_id"])

# left join
silver_with_borough = silver.join(
    loc_slim,
    silver["sensor_id"].cast("long") == loc_slim["loc_sensor_id"],
    "left"
).drop("loc_sensor_id") \
.select(
    silver["sensor_id"],
    "name",
    "borough",
    "latitude",
    "longitude",
    "parameter",
    "value",
    "unit",
    "datetime_from",
    "datetime_to",
    "expected_count",
    "observed_count",
    "percent_complete",
    "percent_coverage",
    "summary_min",
    "summary_max",
    "summary_sd"
)

silver_with_borough.cache()
print(f"✅ Jami qatorlar: {silver_with_borough.count():,}")

silver_with_borough.groupBy("borough") \
    .agg(F.count("*").alias("yozuvlar")) \
    .orderBy("borough") \
    .show()
# ============================================================
# CELL — itransition_project ga saqlash
# ============================================================
silver_with_borough.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_openaq_with_borough")

print("✅ Saqlandi!")
spark.sql("SELECT COUNT(*) FROM silver_openaq_with_borough").show()

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 8, Finished, Available, Finished, False)

✅ Jami qatorlar: 25,496
+-------------+--------+
|      borough|yozuvlar|
+-------------+--------+
|        Bronx|    1392|
|     Brooklyn|    5035|
|    Manhattan|    6977|
|       Queens|    2730|
|Staten Island|    2512|
|      Unknown|    6850|
+-------------+--------+

✅ Saqlandi!
+--------+
|count(1)|
+--------+
|   25496|
+--------+



In [9]:
from pyspark.sql import functions as F

# Zararli parametrlar
zararli_parametrlar = ["pm25", "pm10", "pm1", "um003", "no2", "nox", "co", "o3", "so2", "no"]

# ============================================================
# 1. POLLUTION HOTSPOTS BY LOCATION/TIME
# ============================================================
gold_hotspot = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("soat", F.hour("datetime_from")) \
    .withColumn("sana", F.to_date("datetime_from")) \
    .groupBy("borough", "sensor_id", "name", "latitude", "longitude", "parameter", "sana", "soat") \
    .agg(
        F.round(F.avg("value"), 3).alias("avg_value"),
        F.round(F.max("value"), 3).alias("max_value"),
        F.round(F.min("value"), 3).alias("min_value"),
        F.count("*").alias("kuzatish_soni")
    ) \
    .orderBy(F.desc("avg_value"))

print("=== 1. HOTSPOT ===")
display(gold_hotspot.limit(20))

# Warehousega saqlash
# gold_hotspot.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save("abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_hotspot")

# spark.sql("""
#     CREATE TABLE IF NOT EXISTS gold_hotspot
#     USING DELTA
#     LOCATION 'abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_hotspot'
# """)
# print("✅ gold_hotspot saqlandi!")


# ============================================================
# 2. HIGH-TRAFFIC vs AIR QUALITY KORRELYATSIYA
# ============================================================

# Taxi: borough + sana bo'yicha aggregate
taxi_agg = taxi_with_borough \
    .withColumn("sana", F.to_date("pickup_date")) \
    .groupBy("borough", "sana") \
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )

# OpenAQ: sana ustuni qo'shish
openaq_daily = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("sana", F.to_date("datetime_from"))

# Inner join
traffic_air = openaq_daily.join(
    taxi_agg,
    (openaq_daily["borough"] == taxi_agg["borough"]) &
    (openaq_daily["sana"]    == taxi_agg["sana"]),
    "inner"
).select(
    openaq_daily["borough"],
    openaq_daily["sana"],
    openaq_daily["sensor_id"],
    openaq_daily["parameter"],
    openaq_daily["value"],
    openaq_daily["unit"],
    openaq_daily["datetime_from"],
    openaq_daily["datetime_to"],
    openaq_daily["summary_min"],
    openaq_daily["summary_max"],
    openaq_daily["summary_sd"],
    openaq_daily["expected_count"],
    openaq_daily["observed_count"],
    openaq_daily["percent_complete"],
    openaq_daily["percent_coverage"],
    taxi_agg["trip_count"],
    taxi_agg["avg_fare"],
    taxi_agg["avg_duration"],
    taxi_agg["total_revenue"]
)

# Korrelyatsiya
gold_corr = traffic_air \
    .groupBy("borough", "sana", "parameter") \
    .agg(
        F.sum("trip_count").alias("total_trips"),
        F.sum("value").alias("total_pollution")
    )

print("=== 2. KORRELYATSIYA ===")
gold_corr.agg(
    F.corr("total_trips", "total_pollution").alias("korrelyatsiya")
).show()

# Borough bo'yicha
gold_corr \
    .groupBy("borough") \
    .agg(F.corr("total_trips", "total_pollution").alias("korrelyatsiya")) \
    .orderBy(F.desc("korrelyatsiya")) \
    .show()

# Warehousega saqlash
# gold_corr.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save("abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_traffic_air_corr")

# spark.sql("""
#     CREATE TABLE IF NOT EXISTS gold_traffic_air_corr
#     USING DELTA
#     LOCATION 'abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_traffic_air_corr'
# """)
# print("✅ gold_traffic_air_corr saqlandi!")


# ============================================================
# 3. SEASONAL AIR QUALITY VARIATIONS
# ============================================================
gold_seasonal = silver_with_borough \
    .filter(F.col("parameter").isin(zararli_parametrlar)) \
    .withColumn("oy", F.month("datetime_from")) \
    .withColumn("mavsum", F
        .when(F.month("datetime_from").isin(12, 1, 2), "Qish")
        .when(F.month("datetime_from").isin(3, 4, 5),  "Bahor")
        .when(F.month("datetime_from").isin(6, 7, 8),  "Yoz")
        .otherwise("Kuz")
    ) \
    .groupBy("mavsum", "oy", "borough", "parameter") \
    .agg(
        F.round(F.avg("value"), 3).alias("avg_value"),
        F.round(F.max("value"), 3).alias("max_value"),
        F.round(F.min("value"), 3).alias("min_value"),
        F.count("*").alias("kuzatish_soni")
    ) \
    .orderBy("mavsum", "oy", "borough", "parameter")

print("=== 3. SEASONAL ===")
display(gold_seasonal.limit(20))

# Warehousega saqlash
# gold_seasonal.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save("abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_seasonal")

# spark.sql("""
#     CREATE TABLE IF NOT EXISTS gold_seasonal
#     USING DELTA
#     LOCATION 'abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/gold_seasonal'
# """)
# print("✅ gold_seasonal saqlandi!")

StatementMeta(, 03be0cd1-ef56-4c1b-9fd1-eaa5bdf72d7b, 12, Finished, Available, Finished, False)

=== 1. HOTSPOT ===


SynapseWidget(Synapse.DataFrame, de00d9d9-e6f6-4a69-a1ad-6b7f57f608f6)

=== 2. KORRELYATSIYA ===
+--------------------+
|       korrelyatsiya|
+--------------------+
|-0.04419740751726...|
+--------------------+

+-------------+--------------------+
|      borough|       korrelyatsiya|
+-------------+--------------------+
|    Manhattan|   0.835694271250332|
|        Bronx|  0.7929709054333532|
|Staten Island| 0.46279645568550876|
|       Queens|-0.13361855771694314|
|      Unknown|-0.14108422245325408|
|     Brooklyn| -0.1578420584708651|
+-------------+--------------------+

=== 3. SEASONAL ===


SynapseWidget(Synapse.DataFrame, 50a16b74-c865-47e3-a5d9-a80f219ee8fa)